# Reference Resolution Dev Notebook

Third attempt at integrating methods directly into METER's pytorch lightning framework. This one samples some number of bounding boxes over a max_bb limit to reduce gpu memory usage.

In [20]:
import random
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER
import pandas as pd

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from pytorch_lightning import LightningDataModule

from transformers import ElectraTokenizer, AutoTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

In [2]:
# temporary variable switch between servers, fix before deployment
tensor_book = True
frege = False

if tensor_book:
    data_root =  "/home/claytonfields/nlp/code/vilt/data/arrow"
    load_path = "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt"
    refer_root = "/home/claytonfields/nlp/code/data/coco"
    device = torch.device('cpu')
    num_gpus = 1
else:
    data_root =  "/data/clayton/meter/data/arrow"
    load_path = "/data/clayton/meter/result/meter_electra_small_deit_tiny_p16_is224_bs288_ts1M/checkpoints/epoch=43-step=898039.ckpt"
    refer_root = "/data/clayton/datasets/coco"
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    if frege:
        num_gpus = 2
    else:
        num_gpus = 1


## RefCOCO Data

### Import  Data

In [3]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=8.89s)


In [4]:
refer.IMAGE_DIR = '/home/claytonfields/nlp/code/data/coco/images/mscoco/train2014'

In [5]:
config = copy.deepcopy(_config)

### Find Max Number of Objects

In [6]:
train_ids = refer.getRefIds()
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    size.append(len(objs))
num_refs = len(train_ids)
max_size = np.max(size)
avg_size = np.mean(size)
median_size = np.median(size)
p_70 =  np.percentile(size, 70)
p_80 =  np.percentile(size, 80)
p_90 =  np.percentile(size, 90)
p_95 =  np.percentile(size, 95)
p_99 =  np.percentile(size, 99)
num_over_42 = np.sum(np.array(size) >= 42)

print(f'The most objects in any reference is {max_size}')
print(f'The average number of objects in each reference is {avg_size}')
print(f'The median number of objects in each reference is {median_size}')
print()
print(f'The 70th percentile of the number of objects in each reference is {p_70}')
print(f'The 80th percentile of the number of objects in each reference is {p_80}')
print(f'The 80th percentile of the number of objects in each reference is {p_90}')
print(f'The 95th percentile of the number of objects in each reference is {p_95}')
print(f'The 99th percentile of the number of objects in each reference is {p_99}')
print()
print(f'A max_bb of 42 would exclude {num_over_42} of {num_refs} refs')

The most objects in any reference is 75
The average number of objects in each reference is 10.60916
The median number of objects in each reference is 8.0

The 70th percentile of the number of objects in each reference is 13.0
The 80th percentile of the number of objects in each reference is 16.0
The 80th percentile of the number of objects in each reference is 21.0
The 95th percentile of the number of objects in each reference is 26.0
The 99th percentile of the number of objects in each reference is 41.0

A max_bb of 42 would exclude 453 of 50000 refs


### Find Max Number of Sentences

In [7]:
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    objs = refer.imgToAnns[img_id]
    size.append(len(ref['sentences']))
max_size = max(size)
print(f'The most sentences in any reference is {max_size}')

The most sentences in any reference is 6


In [22]:
_config = {  
    "exp_name":"finetune_ref",
    "seed" : 0,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["coco"],
    "loss_names" :{'itm': 0,
    'mlm': 0,
    'mpp': 0,
    'vqa': 0,
    'vcr': 0,
    'vcr_qar': 0,
    'nlvr2': 0,
    'irtr': 0,
    'contras': 0,
    'snli': 0,
    'ref': 1,
    "mrpc" : 0,
    "rte" : 0,
    'wnli' : 0,
    'sst2' : 0,
    'qqp' : 0,
    'qnli' : 0,
    'mnli' : 0,
    'cola' : 0,
    'cifar10' : 0
    },
    "batch_size" : 10,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.

    # Image setting
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,

    # Text Setting
    "vqav2_label_size" : 3129,
    "max_text_len" : 40,
    "text_encoder" : "google/electra-small-discriminator",
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,

    # Transformer Setting
    "num_cross_layers" : 6,
    "image_encoder_hidden_size" : 192,
    "text_encoder_hidden_size" : 256,
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "num_layers" : 6,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,

    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 1e-5,
    "weight_decay" : 0.01,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 10000,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze_image_encoder' : False,
    'freeze_text_encoder' : False,
    'random_init_text_encoder' : False,
    'random_init_vision_encoder' : False,
    'freeze_cross_modal_layers' : False,
    "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : data_root,
    "log_dir" : "result",
    "per_gpu_batchsize" : 3,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : num_gpus,
    "num_nodes" : 1,
    "load_path" : load_path,
    "num_workers" : 12,
    "precision" : 16
}

In [24]:
config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
# dm = RefcocoDataModule(config, refer, collate_fn)
model = METERTransformerSS(_config)
tokenizer = AutoTokenizer.from_pretrained(config['text_encoder'])
# model.current_tasks.append('ref')

Seed set to 0
Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Data Class for Ref Res with multiple samples

In [26]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, errors, split='', max_bb = 42):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.errors = errors
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            
            ref = self.refer.Refs[ref_id]
            img_id = ref['image_id']
            objs = refer.imgToAnns[img_id]
            if len(objs) <= self.max_bb:
                for sent_id in ref['sent_ids']:
                    if not sent_id in self.errors:
                        sent_ids.append(sent_id)
        return sent_ids
    
    def __getitem__(self, index):
        max_bb = self.max_bb
        
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        obj_pad = [0 for _ in range(max_bb-len(obj_ids))]
        obj_ids_total = obj_ids+obj_pad

        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        
        num_sub_images = len(sub_images)
        num_pad = max_bb - num_sub_images 
        
        pad_image = torch.zeros(1,3,224,224)
        for _ in range(max_bb - num_sub_images):
            sub_images.append(pad_image)
        
        # text ids
        ids = self.tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        repeat_ids = torch.tensor(ids).repeat(num_sub_images,1)
        pad_ids =  torch.zeros(num_pad,40)
        text_ids = torch.cat((repeat_ids, pad_ids)).to(torch.long)
        # text masks
        num_tokens = torch.where(text_ids[0] > 0)[0].size(dim=0)
        masks = torch.cat((torch.ones(num_tokens), torch.zeros(40-num_tokens))).to(torch.long)
        repeat_masks = masks.repeat(num_sub_images,1)
        pad_masks = torch.zeros(num_pad, 40)
        text_masks = torch.cat((repeat_masks, pad_masks)).to(torch.long)
        # text_labels
        labels = torch.full((40,),-100)
        repeat_labels = labels.repeat(num_sub_images, 1)
        pad_labels = torch.zeros(num_pad, 40)
        text_labels = torch.cat((repeat_labels, pad_labels)).to(torch.long)
        
        target = torch.tensor([obj_ids.index(ann_id)])

        return_dict = {
            'ann_id' : ann_id,
            'image' : [torch.cat(sub_images)],#.to(self.device)],
            'obj_ids' : torch.tensor(obj_ids_total),#.to(self.device),
            'target' : target,#.to(self.device),
            'text' : sent['sent'],
            'text_ids' : text_ids,#.to(self.device),
            'text_labels' : text_labels,#.to(self.device),
            'text_masks' : text_masks,#.to(self.device)
        }
        
        return return_dict

In [27]:
errors_df = pd.read_csv('Errors.csv')
errors = errors_df['Sent ID'].to_list()

In [28]:
ds = RefcocoDataset(refer, tokenizer, errors, split='train')
ds

In [29]:
ds[0]

AttributeError: module 'PIL.Image' has no attribute 'ANTIALIAS'

In [12]:
split = 'train'
sent_ids = []
max_bb = 15

for ref_id in refer.getRefIds(split=split):

    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    objs = refer.imgToAnns[img_id]
    
    for sent_id in ref['sent_ids']:
        if not sent_id in errors:
            sent_ids.append(sent_id)
sent_ids

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,


In [13]:
ann_id

598731

In [14]:
index = 0
max_bb = 15

sent_id = sent_ids[index]
ref = refer.sentToRef[sent_id]
sent = refer.Sents[sent_id]

img_id = ref['image_id']
ann_id = ref['ann_id']
objs = refer.imgToAnns[img_id]
obj_ids = [obj['id'] for obj in objs]
#         obj_pad = [0 for _ in range(max_bb-len(obj_ids))]
#         obj_ids_total = obj_ids+obj_pad

num_objs = len(obj_ids)
if num_objs > max_bb:
    obj_ids.remove(ann_id)
#     obj_ids = np.random.choice(obj_ids,max_bb - 1, replace=False).tolist()
    obj_ids = random.choices(obj_ids, k=max_bb-1)
    obj_ids.append(ann_id)
    random.shuffle(obj_ids)
# print(len(obj_ids))
# obj_ids
sub_images = []
for obj_id in obj_ids:
    x_a = get_bounded_subimage(refer, img_id, obj_id, xs=224,ys=224, show=False)
    if x_a is not None:
        sub_images.append(x_a)

num_sub_images = len(sub_images)
print(num_sub_images)
# sub_images[0].shape

num_pad = max_bb - num_sub_images 

#         pad_image = torch.zeros(1,3,224,224)
#         for _ in range(max_bb - num_sub_images):
#             sub_images.append(pad_image)

# text ids
ids = tokenizer.encode(
    sent['sent'],
    padding="max_length",
    truncation=True,
    max_length=40,
    return_special_tokens_mask=True,
)
ids = ids
text_ids = torch.tensor(ids).repeat(num_sub_images,1)
#         pad_ids =  torch.zeros(num_pad,40)
#         text_ids = torch.cat((repeat_ids, pad_ids)).to(torch.long)
# text masks
num_tokens = torch.where(text_ids[0] > 0)[0].size(dim=0)
masks = torch.cat((torch.ones(num_tokens), torch.zeros(40-num_tokens))).to(torch.long)
text_masks = masks.repeat(num_sub_images,1)
#         pad_masks = torch.zeros(num_pad, 40)
#         text_masks = torch.cat((repeat_masks, pad_masks)).to(torch.long)
# text_labels
labels = torch.full((40,),-100)
text_labels = labels.repeat(num_sub_images, 1)
#         pad_labels = torch.zeros(num_pad, 40)
#         text_labels = torch.cat((repeat_labels, pad_labels)).to(torch.long)

target = torch.tensor([obj_ids.index(ann_id)])

return_dict = {
    'ann_id' : ann_id,
    'image' : sub_images,#.to(self.device)],
    'obj_ids' : torch.tensor(obj_ids),#.to(self.device),
    'target' : target,#.to(self.device),
    'num_bb' : num_sub_images,
    'text' : sent['sent'],
    'text_ids' : text_ids,#.to(self.device),
    'text_labels' : text_labels,#.to(self.device),
    'text_masks' : text_masks,#.to(self.device)
}
return_dict

15


NameError: name 'tokenizer' is not defined

In [ ]:
ann = obj_ids.remove(ann_id)
print(len(obj_ids))
obj_ids

In [ ]:
def collate(batch):
    targets = []
    for b in batch:
        targets.append(b['target'])
    targets = torch.tensor(targets)
    
    max_bb = max([b['num_bb'] for b in batch])

    pad_image = torch.zeros(1,3,224,224)
    
    for b in batch:
        num_examples = b['num_bb']
        num_pad = max_bb - num_examples


        for _ in range(num_pad):
            b['image'].append(pad_image)
        b['image'] = [torch.cat(b['image'])]

        pad_masks = torch.zeros(num_pad, 40)
        b['text_masks'] = torch.cat((b['text_masks'], pad_masks)).to(torch.long)


        pad_labels = torch.zeros(num_pad, 40)
        b['text_labels'] = torch.cat((b['text_labels'], pad_labels)).to(torch.long)

        pad_ids =  torch.zeros(num_pad,40)
        b['text_ids'] = torch.cat((b['text_ids'], pad_ids)).to(torch.long)
    
        return (batch, targets)
    

In [ ]:
errors_df = pd.read_csv('Errors.csv')
errors_list = errors_df['Sent ID'].to_list()
# dm = RefcocoDataModule(config, refer, device, errors_list, collate)
ds = RefcocoDataset(refer, tokenizer, device, errors_list, split='train', max_bb=42)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate
                }
training_loader = torch.utils.data.DataLoader(ds, **train_params)

In [ ]:
batch = next(iter(training_loader))

In [ ]:
targets = batch[1]
batch = batch[0]

In [ ]:
len(batch)

In [ ]:
batch[5]['image'].__len__()

In [ ]:
batch[0]['image'].__len__()

In [ ]:
max_bb = max([b['num_bb'] for b in batch])

pad_image = torch.zeros(1,3,224,224)
ret = []
for b in batch:
    num_examples = b['num_bb']
    num_pad = max_bb - num_examples


    for _ in range(num_pad):
        b['image'].append(pad_image)
    b['image'] = [torch.cat(b['image'])]

    pad_masks = torch.zeros(num_pad, 40)
    b['text_masks'] = torch.cat((b['text_masks'], pad_masks)).to(torch.long)


    pad_labels = torch.zeros(num_pad, 40)
    b['text_labels'] = torch.cat((b['text_labels'], pad_labels)).to(torch.long)

    pad_ids =  torch.zeros(num_pad,40)
    b['text_ids'] = torch.cat((b['text_ids'], pad_ids)).to(torch.long)

    

In [ ]:
batch[-1]

**Data Module for pytorch lightning**

## METER Model

In [15]:
config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
dm = RefcocoDataModule(config, refer, collate_fn)
model = METERTransformerSS(_config)
# model.current_tasks.append('ref')

Global seed set to 0


NameError: name 'RefcocoDataModule' is not defined

## Perform Ref Res with METER

**Define key variables and parameters**

In [ ]:
optim = AdamW(model.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 10


epochs = 1
# loader = dm.train_dataloader()
optim = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
dvice  = torch.device('cpu')

In [ ]:
ds = RefcocoDataset(refer, tokenizer, split='train', max_bb=42)
# ds = NewRefcocoDataset(refer, tokenizer)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate
                }

training_loader = torch.utils.data.DataLoader(ds, **train_params)

In [ ]:
model.current_tasks = ['ref']

In [ ]:
logits

### Training

**Training Loop Dev Cell**

In [ ]:
pl.seed_everything(_config["seed"])

# dm = MTDataModule(_config, dist=False)

model = METERTransformerSS(_config)
exp_name = f'{_config["exp_name"]}'

os.makedirs(_config["log_dir"], exist_ok=True)
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    save_top_k=1,
    verbose=True,
    monitor="val/the_metric",
    mode="max",
    save_last=True,
)
logger = pl.loggers.TensorBoardLogger(
    _config["log_dir"],
    name=f'{exp_name}_seed{_config["seed"]}_from_{_config["load_path"].split("/")[-1][:-5]}',
)

lr_callback = pl.callbacks.LearningRateMonitor(logging_interval="step")
callbacks = [checkpoint_callback, lr_callback]

num_gpus = (
    _config["num_gpus"]
    if isinstance(_config["num_gpus"], int)
    else len(_config["num_gpus"])
)

grad_steps = max(_config["batch_size"] // (
    _config["per_gpu_batchsize"] * num_gpus * _config["num_nodes"]
), 1)

max_steps = _config["max_steps"] if _config["max_steps"] is not None else None

trainer = pl.Trainer(
    gpus=0,
    num_nodes=_config["num_nodes"],
    precision=_config["precision"],
#     accelerator="cpu",
    benchmark=True,
    deterministic=True,
    max_epochs=_config["max_epoch"] if max_steps is None else 1000,
    max_steps=max_steps,
    callbacks=callbacks,
    logger=logger,
    #prepare_data_per_node=False,
    #replace_sampler_ddp=False,
    accumulate_grad_batches=grad_steps,
    log_every_n_steps=10,
    flush_logs_every_n_steps=10,
#     resume_from_checkpoint=_config["resume_from"],
    weights_summary="top",
    fast_dev_run=_config["fast_dev_run"],
    val_check_interval=_config["val_check_interval"],
)

if not _config["test_only"]:
    trainer.fit(model, datamodule=dm)
else:
    trainer.test(model, datamodule=dm)

Global seed set to 0
GPU available: True, used: False
TPU available: False, using: 0 TPU cores
/home/claytonfields/anaconda3/envs/vl/lib/python3.7/site-packages/pytorch_lightning/utilities/distributed.py:69: UserWarning: GPU available but not used. Set the gpus flag in your trainer `Trainer(gpus=1)` or script `--gpus=1`.
  warnings.warn(*args, **kwargs)

   | Name                        | Type              | Params
-------------------------------------------------------------------
0  | cross_modal_text_transform  | Linear            | 65.8 K
1  | cross_modal_image_transform | Linear            | 49.4 K
2  | token_type_embeddings       | Embedding         | 512   
3  | vit_model                   | VisionTransformer | 5.5 M 
4  | text_transformer            | ElectraModel      | 13.5 M
5  | cross_modal_image_layers    | ModuleList        | 6.3 M 
6  | cross_modal_text_layers     | ModuleList        | 6.3 M 
7  | cross_modal_image_pooler    | Pooler            | 65.8 K
8  | cross_modal_

Validation sanity check: 0it [00:00, ?it/s]

In [ ]:
batch = next(iter(training_loader))

In [ ]:
model(batch)

**TODO:** Work on training that is compatible with METER's pytorch lightning cofiguration

In [ ]:
model.train()
# losses = []
# logit_list = []
# targets = []
optim.zero_grad()
for batch in training_loader:
    try:
        model(batch)
        
    except RuntimeError:
        print(f'RuntimeError')
loss = loss_fn(logits.reshape(1,-1),target)
losses.append(loss.item())
loss.backward()
optim.step()

## Test Cells:

In [27]:
model.current_tasks = vqa 

[]

In [23]:
model

METERTransformerSS(
  (cross_modal_text_transform): Linear(in_features=256, out_features=256, bias=True)
  (cross_modal_image_transform): Linear(in_features=192, out_features=256, bias=True)
  (token_type_embeddings): Embedding(2, 256)
  (vit_model): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_featu